In [1]:
import gpt as g
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from itertools import permutations

import scipy
import math
from scipy import special as sp
import time

SharedMemoryNone: SharedMemoryAllocate 1073741824 GPU implementation 
0SharedMemoryNone:  SharedMemoryNone.cc acceleratorAllocDevice 1073741824bytes at 0x300000000 for comms buffers 

__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|_ |  |  |  |  |  |  |  |  |  |  |  | _|__
__|_                                    _|__
__|_   GGGG    RRRR    III    DDDD      _|__
__|_  G        R   R    I     D   D     _|__
__|_  G        R   R    I     D    D    _|__
__|_  G  GG    RRRR     I     D    D    _|__
__|_  G   G    R  R     I     D   D     _|__
__|_   GGGG    R   R   III    DDDD      _|__
__|_                                    _|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
  |  |  |  |  |  |  |  |  |  |  |  |  |  |  


Copyright (C) 2015 Peter Boyle, Azusa Yamaguchi, Guido Cossu, Antonin Portelli and other authors

This program is free software; you can redistribute it and/or modify
it under the term

In [3]:
size = 8
grid = g.grid([size, size, size, size], g.double)
rng = g.random(str(time.time()))

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)

action_gauge = g.qcd.gauge.action.wilson(7.0)

metro = g.algorithms.markov.metropolis(rng)
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()
pure_gauge = True


# thermalize lattice first
beta = g.default.get_float("--beta", 7.0)
seed = g.default.get("--seed", "hmc-pure-gauge")
ntherm = g.default.get_int("--ntherm", 10)
n = g.default.get_int("--n", 200)
nwrite = g.default.get_int("--nwrite", 10)
g.default.set_verbose("omf4")

# conjugate momenta
mom = g.group.cartesian(U)

# Log
g.message(f"Lattice = {grid.fdimensions}")
g.message("Actions:")
# action for conj. momenta
a0 = g.qcd.scalar.action.mass_term()
g.message(f" - {a0.__name__}")

# wilson action 
a1 = g.qcd.gauge.action.wilson(beta)
g.message(f" - {a1.__name__}")
    
    
def hamiltonian():
    return a0(mom) + a1(U) 

# molecular dynamics
sympl = g.algorithms.integrator.symplectic
    
iphmc = sympl.update_p(mom, lambda: a1.gradient(U, U))
iqhmc = sympl.update_q(U, lambda: a0.gradient(mom, mom))

# integrator
mdint_hmc = sympl.leap_frog(50, iphmc, iqhmc)
#g.message(f"Integration scheme:\n{mdint}")
    
# metropolis
metro = g.algorithms.markov.metropolis(rng)
    
# MD units
tau = 1.0
g.message(f"tau = {tau} MD units")

def hmc(tau, mom):
    rng.normal_element(mom)
    accrej = metro(U)
    h0 = hamiltonian()
    mdint_hmc(tau)
    h1 = hamiltonian()
    return [accrej(h1, h0), h1 - h0]


# thermalization
for i in range(1, 20):
    h = []
    timer = g.timer("hmc")
    for _ in range(ntherm // 10):
        timer("trajectory")
        h += [hmc(tau, mom)]
    h = np.array(h)
    timer()
    g.message(f"{i*10} % of thermalization completed")
    g.message(timer)
    g.message(
        f"Plaquette = {g.qcd.gauge.plaquette(U)}, Acceptance = {np.mean(h[:,0]):.2f}, |dH| = {np.mean(np.abs(h[:,1])):.4e}"
    )


start = time.time()

# production
history = []
plaq = []


wf_energyHMC = []
for i in range(50):
    history += [hmc(tau, mom)]
    P = g.qcd.gauge.plaquette(U)
    plaq.append(P)
    g.message(f"Trajectory {i}, P={P}")


end = time.time()
print("time taken = ", end - start)

history = np.array(history)
g.message(f"Acceptance rate = {np.mean(history[:,0]):.2f}")
g.message(f"<|dH|> = {np.mean(np.abs(history[:,1])):.4e}")



GPT :      45.816459 s : Initializing gpt.random(1758140616.803833,vectorized_ranlux24_389_64) took 0.000519037 s
GPT :      45.862741 s : Lattice = [8, 8, 8, 8]
GPT :      45.863239 s : Actions:
GPT :      45.863591 s :  - mass_term(m^-1 = 1.0)
GPT :      45.863986 s :  - wilson(7.0)
GPT :      45.864701 s : tau = 1.0 MD units
GPT :      46.524286 s : 10 % of thermalization completed
GPT :      46.524592 s : hmc:
                       : trajectory           6.59e-01 s (= 100.00 %); time/s = 6.59e-01/6.59e-01/6.59e-01 (min/max/avg)
GPT :      46.525241 s : Plaquette = 0.29453611118208933, Acceptance = 1.00, |dH| = 1.3102e+01
GPT :      47.163078 s : 20 % of thermalization completed
GPT :      47.163362 s : hmc:
                       : trajectory           6.38e-01 s (= 100.00 %); time/s = 6.38e-01/6.38e-01/6.38e-01 (min/max/avg)
GPT :      47.163963 s : Plaquette = 0.40203099652200397, Acceptance = 1.00, |dH| = 6.7454e+00
GPT :      47.793297 s : 30 % of thermalization completed
GPT 

In [4]:
W = g.copy(U)
V = g.copy(U)

In [30]:
action = g.qcd.gauge.action.wilson(7)
print("UNFLOWED", a1(U))

lsm = g.qcd.gauge.smear.local_stout(rho=0.05, dimension=1, checkerboard=g.even)
action_sm = action.transformed(lsm)
action_sm.assert_gradient_error(rng, W, W, 1e-3, 1e-7)
lsm.assert_log_det_jacobian(U, 1e-5, (2, 2, 2, 0), 1e-7)

action_log_det = lsm.action_log_det_jacobian()
action_log_det.assert_gradient_error(rng, W, W, 1e-3, 1e-8)


UNFLOWED 56771.62349213661
GPT :    1357.682119 s : Test that functional is real: 0.0
GPT :    1357.794521 s : Assert gradient error: 7.568299999032016e-11 < 1e-07
GPT :    1360.988711 s : assert_log_det_jacobian: 1.8537171797561314e-11 < 1e-07 ; log_det = (-2.0730253683467508+0j) , log_det_appx = -2.073025368365288
GPT :    1361.025474 s : Test that functional is real: 0.0
GPT :    1361.402961 s : Assert gradient error: 2.126234676678763e-12 < 1e-08


In [31]:
sum(lsm.log_det_jacobian(W)[:])

array([-3756.33166166+0.j])

In [32]:
t = 0.0
eps = 0.05
U_wf = g.copy(U)
        
U_wf = g.qcd.gauge.smear.wilson_flow(U_wf, epsilon=eps)

a1(U_wf)

np.float64(35054.76755631529)